# 1. Import Libraries

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Load Data

In [0]:
spark_df = spark.sql("select * from brightlearn.brighttv.user_profile")
df_user = spark_df.toPandas()

df_user.head()

In [0]:
spark_df1 = spark.sql("select * from brightlearn.brighttv.viewership")
df_views = spark_df1.toPandas()

df_views.head()

# 3. Data Pre-processing:

In [0]:
df_user.info()

In [0]:
df_views.info()

## Handling missing values.

In [0]:
print(df_user.isnull().sum())

In [0]:
print(df_views.isnull().sum())

## Handling missing data

In [0]:
df_user["Province"] = df_user.filna("Gauteng")

# Data Cleaning:

## Remove duplicates

In [0]:
df_user = df_user.drop_duplicates()

In [0]:
df_views.duplicated()


In [0]:
print("Number of duplicated rows: ",df_views.duplicated().sum())

In [0]:
display(df_views[df_views.duplicated() ==True])

In [0]:
print(df_views[df_views.duplicated()])

In [0]:
df_views = df_views.drop_duplicates()

## Fix inconsistencies

In [0]:
df_views['UserID'] = df_views['UserID'].astype(str)
df_user['UserID'] = df_user['UserID'].astype(str)

In [0]:
df_views.head()

In [0]:
# Fix Datetime Columns
df_views['start_time'] = pd.to_datetime(df_views['RecordDate2'], format='%m/%d/%y %H:%M')

df_views.head(100)

In [0]:
# Convert date timestamp to actual timestamp
df_views['duration'] = pd.to_timedelta(df_views['Duration 2'].dt.hour, unit='h') + \
                       pd.to_timedelta(df_views['Duration 2'].dt.minute, unit='m') + \
                       pd.to_timedelta(df_views['Duration 2'].dt.second, unit='s')


df_views['duration'] = pd.to_timedelta(df_views['duration'] )
df_views.head(20)

In [0]:
# Convert Duration to Minutes
df_views['session_duration'] = df_views['duration'].dt.total_seconds() / 60

df_views.head(20)

In [0]:
# Create End Time
df_views['end_time'] = df_views['start_time'] + df_views['duration']
df_views.head(20)

In [0]:
# Convert to SAST
df_views['start_time_sast'] = df_views['start_time'] + pd.Timedelta(hours=2)
df_views['end_time_sast'] = df_views['end_time'] + pd.Timedelta(hours=2)

df_views.head(20)

In [0]:
df_views['hour'] = df_views['start_time_sast'].dt.hour
df_views['day'] = df_views['start_time_sast'].dt.day_name()
df_views['date'] = df_views['start_time_sast'].dt.date

df_views['is_weekend'] = df_views['day'].isin(['Saturday', 'Sunday'])

# 4. Data Transformation:

Filter, sort & Modify

In [0]:
# Channel-Level Analysis
channel_usage = df_views.groupby('Channel2')['session_duration'].mean().sort_values(ascending=False).reset_index()

sns.barplot(data=channel_usage.head(10), x='session_duration', y='Channel2')
plt.title("Top Content by Avg Watch Time")
plt.show()

In [0]:
def categorize_content(x):
    if "Cricket" in x:
        return "Sports"
    elif "News" in x:
        return "News"
    else:
        return "Entertainment"

df_views['content_category'] = df_views['Channel2'].apply(categorize_content)

# 5. Data Visualization:

## 6. Represent data graphically.